In [1]:
import argparse
import warnings
from concurrent.futures import ProcessPoolExecutor

import sys
sys.path.append('/g/data/w28/yk8692/sdmbc_v2/modules')

# import dask  # type: ignore
import numpy as np  # type: ignore
# import pandas as pd # type: ignore
import xarray as xr  # type: ignore
from config import config  # type: ignore
from dask.distributed import Client  # type: ignore

from analysis_plot import AnalysisBC  # type: ignore

# import yaml  # type: ignore
from bc_grid_function import bc_correction_grid_cell_hist  # type: ignore
from data_preparation import (
    assign_w_6hr,
    assign_w_day,
    convert_to_daily_with_fraction,
    extract_and_reshape_delayed,
    generate_file_paths,
    generate_file_paths_obs,
    load_preprocess_variable,
    validate_inputs,
)
from output_reformatter import reformat_and_save_2d  # type: ignore
from output_reformatter import reformat_and_save_3d  # type: ignore

warnings.simplefilter("ignore", UserWarning)

In [2]:
# Start of the script -----------------------------------------------------
def setup_client(n_workers=None, threads_per_worker=None):
    """
    Set up Dask client for parallel processing, allowing customization of workers and threads.

    Args:
        n_workers (int): Number of workers to use.
        threads_per_worker (int): Number of threads per worker.

    Returns:
        Client: A Dask distributed client instance.
    """
    if n_workers is None or threads_per_worker is None:
        c = Client()
    else:
        c = Client(n_workers=n_workers, threads_per_worker=threads_per_worker)
    print("Dask client setup complete.")
    return c


def parse_arguments():
    """
    Parse command-line arguments.

    Returns:
        argparse.Namespace: Parsed arguments.
    """
    parser = argparse.ArgumentParser(description="Run atmospheric data interpolation.")
    parser.add_argument(
        "--config",
        type=str,
        default="./user_input_test.yaml",
        help="Path to the YAML configuration file.",
    )
    parser.add_argument(
        "--var",
        type=str,
        default=["hus", "ta", "ua", "va"],
        help="Path to the YAML configuration file.",
    )
    # parser.add_argument(
    #     "--nc",
    #     type=int,
    #     default=48,
    #     help="Number of cores to use.",
    # )

    parser.add_argument("--sy", type=int, default=1982, help="Start year.")
    parser.add_argument("--ey", type=int, default=2012, help="End year.")

    return parser.parse_args()

In [3]:
setup_client()

2024-08-08 17:54:13,017 - distributed.preloading - INFO - Creating preload: /g/data/hh5/public/apps/dask-optimiser/schedplugin.py
2024-08-08 17:54:13,020 - distributed.utils - INFO - Reload module schedplugin from .py file
2024-08-08 17:54:13,024 - distributed.preloading - INFO - Import preload module: /g/data/hh5/public/apps/dask-optimiser/schedplugin.py


Modifying workers
Dask client setup complete.


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /node/gadi-cpu-bdw-0413.gadi.nci.org.au/31126/proxy/8787/status,
Dashboard: /node/gadi-cpu-bdw-0413.gadi.nci.org.au/31126/proxy/8787/status,Workers: 28
Total threads: 28,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43895,Workers: 28
Dashboard: /node/gadi-cpu-bdw-0413.gadi.nci.org.au/31126/proxy/8787/status,Total threads: 28
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45227,Total threads: 1
Dashboard: /node/gadi-cpu-bdw-0413.gadi.nci.org.au/31126/proxy/40109/status,Memory: 0 B
Nanny: tcp://127.0.0.1:34213,


In [4]:
startyear_h = config.startyear_h
endyear_h = config.endyear_h
no_of_variables = config.no_of_variables
startyear_h = config.startyear_h
bc_boundary = config.bc_boundary
bc_hist_path = config.bc_hist_path
lat_max = config.lat_max
lat_min = config.lat_min
lon_max = config.lon_max
lon_min = config.lon_min
tlevel = config.tlevel
obs_path = config.obs_path
out_path = config.out_path
input_model = config.input_model
infor = config.infor
gname = config.gname
period = config.period
cinfor = config.cinfor
sinfor = config.sinfor
version = config.version
input_variables = config.target_variable

# Validate inputs
validate_inputs(lat_min, lat_max)
var_list_w = ["w", "ta", "hus"] if bc_boundary == "lateral" else ["tos"]

# List of variables to generate file paths for
variables = input_variables

# Define parameters for selection
lat_range = (lat_min, lat_max)  # Adjust as needed
lon_range = (lon_min, lon_max)  # Adjust as needed
lat_range = (lat_min, lat_max)  # Adjust as needed
lon_range = (lon_min, lon_max)  # Adjust as needed

In [5]:
print(startyear_h, endyear_h, lat_range, lon_range, input_model)

1959 1989 (-85, 85) (0, 360) reanalysis


In [6]:
%%time

sliced_gcm_all = []
sliced_obs_all = []

for level in range(0, 1):
    # =============== Load GCM ===============
    # Generate file paths for each variable
    file_paths_by_variable = {}
    for variable in variables:
        file_paths_by_variable[variable] = generate_file_paths(
            bc_hist_path,
            variable,
            infor,
            gname,
            period,
            cinfor,
            sinfor,
            version,
            startyear_h,
            endyear_h,
        )        
    
    sliced_gcm = xr.Dataset()

    # Load each variable and adjust longitude for ua and va if necessary
    for var_name, file_paths in file_paths_by_variable.items():
        data_var = load_preprocess_variable(
            file_paths, var_name, level, lat_range, lon_range
        )
        # Check if the variable is one of the wind components with different lon
        if var_name in ['ua', 'va']:
            # Let's assume hus and ta have the target longitude values, and they are already loaded
            target_lon = sliced_gcm.lon if 'lon' in sliced_gcm else data_var.lon
            target_lat = sliced_gcm.lat if 'lat' in sliced_gcm else data_var.lat
            target_lev = sliced_gcm.lev if 'lev' in sliced_gcm else data_var.lev

            # Interpolate va to match the target latitude grid
            if var_name == 'va' and 'lat' in data_var.coords and 'lat' in target_lat.coords:
                data_var = data_var.interp(lat=target_lat)

            # Assign the adjusted longitude values to ua or va
            data_var = data_var.assign_coords(
                lon = target_lon, lat = target_lat, lev = target_lev
            )
        # Add the processed variable to the dataset
        sliced_gcm[var_name] = data_var
    sliced_gcm_all.append(sliced_gcm)
    
    # print(sliced_gcm_all)
    # print(sliced_gcm)
    # Rename variables
    # rename_dict = {"hus": "q", "ta": "t", "ua": "u", "va": "v"}
    sliced_gcm = sliced_gcm.sel(
        time=slice(f"{startyear_h}-01-01", f"{endyear_h}-12-31")
    )
    assign_gcm = assign_w_6hr(sliced_gcm, bc_boundary)
    # assign_gcm = assign_gcm[["w", "ta", "hus"]]
    daily_gcm, fraction_factors_gcm = convert_to_daily_with_fraction(assign_gcm)
    daily_gcm_rechunk = daily_gcm.chunk({"time": -1, "lat": "auto", "lon": "auto"})
    print("start delayed process for targets, takes up to 20 mins")
    reshaped_gcm_delayed = extract_and_reshape_delayed(
        daily_gcm_rechunk, no_of_variables, startyear_h, endyear_h
    )

    # =============== Load GCM end ===============

    # =============== Load Obs ===============
    # config should also consider the input and target path
    file_paths_by_variable = {}
    if input_model == "reanalysis":
        for variable in variables:
            file_paths_by_variable[variable] = generate_file_paths_obs(
                obs_path,
                variable,
                gname,
                startyear_h,
                endyear_h,
            )
    else:
        for variable in variables:
            file_paths_by_variable[variable] = generate_file_paths(
                bc_hist_path,
                variable,
                infor,
                gname,
                period,
                cinfor,
                sinfor,
                version,
                startyear_h,
                endyear_h,
            )

    sliced_obs = xr.Dataset()

    # Load each variable and adjust longitude for ua and va if necessary
    for var_name, file_paths in file_paths_by_variable.items():
        obs_var = load_preprocess_variable(
            file_paths, var_name, level, lat_range, lon_range
        )
        # Check if the variable is one of the wind components with different lon
        # if var_name in ['ua', 'va']:
        #     # Let's assume hus and ta have the target longitude values, and they are already loaded
        #     target_lon = sliced_obs.lon if 'lon' in sliced_obs else obs_var.lon
        #     target_lat = sliced_obs.lat if 'lat' in sliced_obs else obs_var.lat
        #     target_lev = sliced_obs.lev if 'lev' in sliced_obs else obs_var.lev

        #     # Interpolate va to match the target latitude grid
        #     if var_name == 'va' and 'lat' in obs_var.coords and 'lat' in target_lat.coords:
        #         obs_var = obs_var.interp(lat=target_lat)

        #     # Assign the adjusted longitude values to ua or va
        #     obs_var = obs_var.assign_coords(
        #         lon = target_lon, lat = target_lat, lev = target_lev
        #     )
        # Add the processed variable to the dataset
        sliced_obs[var_name] = obs_var
    sliced_obs_all.append(sliced_obs)
    
    # Rename variables
    sliced_obs = sliced_obs.sel(
        time=slice(f"{startyear_h}-01-01", f"{endyear_h}-12-31")
    )

    assign_obs = assign_w_6hr(sliced_obs, bc_boundary)
    daily_obs, fraction_factors_obs = convert_to_daily_with_fraction(assign_obs)
    daily_obs_rechunk = daily_obs.chunk({"time": -1, "lat": "auto", "lon": "auto"})
    print("start delayed process for obs, takes up to 20 mins")
    reshaped_obs_delayed = extract_and_reshape_delayed(
        daily_obs_rechunk, no_of_variables, startyear_h, endyear_h
    )
    
    # =============== Load obs end ===============

start delayed process for targets, takes up to 20 mins
start delayed process for obs, takes up to 20 mins


2024-08-08 18:35:08,958 - distributed.worker - ERROR - failed during get data with tcp://127.0.0.1:43737 -> tcp://127.0.0.1:43903
Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.01/lib/python3.10/site-packages/tornado/iostream.py", line 861, in _read_to_buffer
    bytes_read = self.read_from_fd(buf)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.01/lib/python3.10/site-packages/tornado/iostream.py", line 1116, in read_from_fd
    return self.socket.recv_into(buf, len(buf))
TimeoutError: [Errno 110] Connection timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.01/lib/python3.10/site-packages/distributed/worker.py", line 1783, in get_data
    response = await comm.read(deserializers=serializers)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.01/lib/python3.10/site-packages/distributed/comm/

CPU times: user 29min 31s, sys: 7min 54s, total: 37min 25s
Wall time: 47min 33s


In [9]:
print(sliced_obs)

<xarray.Dataset> Size: 18GB
Dimensions:  (time: 45292, lat: 129, lon: 192)
Coordinates:
  * time     (time) datetime64[ns] 362kB 1959-01-01 ... 1989-12-31T18:00:00
  * lat      (lat) float64 1kB -80.0 -78.75 -77.5 -76.25 ... 77.5 78.75 80.0
  * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
    lev      float64 8B 20.0
Data variables:
    hus      (time, lat, lon) float32 4GB dask.array<chunksize=(1, 129, 192), meta=np.ndarray>
    ta       (time, lat, lon) float32 4GB dask.array<chunksize=(1, 129, 192), meta=np.ndarray>
    ua       (time, lat, lon) float32 4GB dask.array<chunksize=(1, 129, 192), meta=np.ndarray>
    va       (time, lat, lon) float32 4GB dask.array<chunksize=(1, 129, 192), meta=np.ndarray>


In [ ]:
%%time
nan_values = sliced_obs.isnull()
nan_summary = nan_values.sum(dim=["time"])
print(nan_summary.compute())

In [16]:
reshaped_gcm_delayed_sel = reshaped_gcm_delayed.sel(lat=slice(-85, 85))
reshaped_obs_delayed_sel = reshaped_obs_delayed.sel(lat=slice(-85, 85))
fraction_factors_gcm_sel = fraction_factors_gcm.sel(lat=slice(-85, 85))
fraction_factors_obs_sel = fraction_factors_obs.sel(lat=slice(-85, 85))
daily_gcm_sel = daily_gcm.sel(lat=slice(-85, 85))
sliced_gcm_sel = sliced_gcm.sel(lat=slice(-85, 85))
sliced_obs_sel = sliced_obs.sel(lat=slice(-85, 85))

In [17]:
test = sliced_obs.compute()

In [18]:
nan_values = test.isnull()
nan_summary = nan_values.sum(dim=["time"])
print(nan_summary.values)

<bound method Mapping.values of <xarray.Dataset> Size: 894kB
Dimensions:  (lat: 145, lon: 192)
Coordinates:
  * lat      (lat) float64 1kB -90.0 -88.75 -87.5 -86.25 ... 87.5 88.75 90.0
  * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
    lev      float64 8B 20.0
Data variables:
    hus      (lat, lon) int64 223kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    ta       (lat, lon) int64 223kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    ua       (lat, lon) int64 223kB 45292 45292 45292 ... 45292 45292 45292
    va       (lat, lon) int64 223kB 45292 45292 45292 ... 45292 45292 45292>


In [20]:
print(obs_var)
nan_values = obs_var.isnull()
nan_summary = nan_values.sum(dim=["time"])
print(nan_summary.values)

<xarray.DataArray 'va' (time: 46752, lat: 144, lon: 192)> Size: 5GB
dask.array<getitem, shape=(46752, 144, 192), dtype=float32, chunksize=(31, 144, 192), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 374kB 1958-01-01T06:00:00 ... 1990-01-01
    lev      float64 8B 9.998
  * lat      (lat) float64 1kB -89.38 -88.12 -86.88 -85.62 ... 86.88 88.12 89.38
  * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
Attributes:
    standard_name:  northward_wind
    long_name:      Northward Wind
    comment:        Meridional wind (positive in a northward direction).
    units:          m s-1
    cell_methods:   time: point
    history:        2019-11-15T04:49:54Z altered by CMOR: replaced missing va...
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [7]:
print(reshaped_gcm_delayed)
print(reshaped_obs_delayed)

<xarray.Dataset> Size: 7GB
Dimensions:  (year: 31, month: 12, day: 31, lat: 137, lon: 192)
Coordinates:
  * year     (year) int64 248B 1959 1960 1961 1962 1963 ... 1986 1987 1988 1989
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * day      (day) int64 248B 1 2 3 4 5 6 7 8 9 ... 23 24 25 26 27 28 29 30 31
  * lat      (lat) float64 1kB -85.0 -83.75 -82.5 -81.25 ... 82.5 83.75 85.0
  * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
Data variables:
    w        (year, month, day, lat, lon) float64 2GB nan nan nan ... nan nan
    ta       (year, month, day, lat, lon) float64 2GB 247.7 247.7 ... 244.2
    hus      (year, month, day, lat, lon) float64 2GB 0.5086 0.5082 ... 0.274
<xarray.Dataset> Size: 7GB
Dimensions:  (year: 31, month: 12, day: 31, lat: 137, lon: 192)
Coordinates:
  * year     (year) int64 248B 1959 1960 1961 1962 1963 ... 1986 1987 1988 1989
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * day      (day) int64 248B 1 

In [14]:
# Initialize empty arrays to hold the results.
dims_time, dims_lat, dims_lon = (
    daily_gcm_sel["time"].shape[0],
    len(daily_gcm_sel.lat),
    len(daily_gcm_sel.lon),
)
bc_params_array = np.empty((dims_lat, dims_lon), dtype=object)

# List to store the results for each grid cell
corrected_data_dict = {}

def process_cell(lat_lon):
    i, j = lat_lon
    lat, lon = (
        reshaped_gcm_delayed_sel.lat.values[i],
        reshaped_gcm_delayed_sel.lon.values[j],
    )
    bc_corrected_gcm_hist, bc_params = bc_correction_grid_cell_hist(
        lat,
        lon,
        reshaped_gcm_delayed_sel,
        reshaped_obs_delayed_sel,
        fraction_factors_gcm_sel,
        fraction_factors_obs_sel,
        var_list_w,
        sliced_gcm_sel,
    )
    return (
        i,
        j,
        bc_corrected_gcm_hist,
        bc_params.to_dict(),
    )  # Assuming to_dict() is implemented

# Prepare lat/lon pairs for processing
lat_lon_pairs = [(i, j) for i in range(dims_lat) for j in range(dims_lon)]

# Use a process pool to parallelize
with ProcessPoolExecutor() as executor:
    results = list(executor.map(process_cell, lat_lon_pairs))

# Store results
for i, j, bc_corrected_gcm_hist, bc_params_dict in results:
    bc_params_array[i, j] = bc_params_dict  # Storing as dict for serialization
    corrected_data_dict[
        (reshaped_gcm_delayed_sel.lat.values[i], reshaped_gcm_delayed_sel.lon.values[j])
    ] = bc_corrected_gcm_hist.assign_coords(
        lat=reshaped_gcm_delayed_sel.lat.values[i],
        lon=reshaped_gcm_delayed_sel.lon.values[j],
    ).expand_dims(
        ["lat", "lon"]
    )

# Save the BC model
np.save(
    f"{out_path}/bc_params_{gname}_to_{input_model}_{startyear_h}_{endyear_h}.npy",
    bc_params_array,
)

corrected_data_list = list(corrected_data_dict.values())

# Combine the DataArrays into a single Dataset
ds_corrected = xr.combine_by_coords(corrected_data_list)

# Set attributes
ds_corrected.attrs["history"] = "Bias-corrected data"

ZeroDivisionError: float division by zero

In [11]:
reshaped_gcm_delayed.to_netcdf(
                f"/g/data/w28/yk8692/sdmbc_v2/tests/delayed_gcm_{level}_{infor}_{gname}_{period}_{cinfor}_{sinfor}_{version}.nc"
            )
reshaped_obs_delayed.to_netcdf(
                f"/g/data/w28/yk8692/sdmbc_v2/tests/delayed_obs_{level}_{infor}_{gname}_{period}_{cinfor}_{sinfor}_{version}.nc"
            )

In [9]:
print(reshaped_gcm_delayed.sel(
            lat=slice(-85, 85),
            lon=slice(config.lon_min, config.lon_max),
        ))
print(reshaped_obs_delayed)

<xarray.Dataset> Size: 7GB
Dimensions:  (year: 31, month: 12, day: 31, lat: 137, lon: 192)
Coordinates:
  * year     (year) int64 248B 1959 1960 1961 1962 1963 ... 1986 1987 1988 1989
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * day      (day) int64 248B 1 2 3 4 5 6 7 8 9 ... 23 24 25 26 27 28 29 30 31
  * lat      (lat) float64 1kB -85.0 -83.75 -82.5 -81.25 ... 82.5 83.75 85.0
  * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
Data variables:
    w        (year, month, day, lat, lon) float64 2GB 38.46 41.63 ... 46.09
    ta       (year, month, day, lat, lon) float64 2GB 247.7 247.7 ... 244.2
    hus      (year, month, day, lat, lon) float64 2GB 0.5086 0.5082 ... 0.274
<xarray.Dataset> Size: 8GB
Dimensions:  (year: 31, month: 12, day: 31, lat: 145, lon: 192)
Coordinates:
  * year     (year) int64 248B 1959 1960 1961 1962 1963 ... 1986 1987 1988 1989
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * day      (day) int64 248B 1 2 

In [10]:
%%time
for level in range(0, 1):
    # =============== Load GCM ===============
    # Generate file paths for each variable
    file_paths_by_variable = {}
    for variable in variables:
        file_paths_by_variable[variable] = generate_file_paths(
            bc_hist_path,
            variable,
            infor,
            gname,
            period,
            cinfor,
            sinfor,
            version,
            startyear_h,
            endyear_h,
        )
    
    sliced_gcm = xr.Dataset()

    # Load each variable and adjust longitude for ua and va if necessary
    for var_name, file_paths in file_paths_by_variable.items():
        data_var = load_preprocess_variable(
            file_paths, var_name, level, lat_range, lon_range
        )
        # Check if the variable is one of the wind components with different lon
        if var_name in ['ua', 'va']:
            # Let's assume hus and ta have the target longitude values, and they are already loaded
            target_lon = sliced_gcm.lon if 'lon' in sliced_gcm else data_var.lon
            target_lat = sliced_gcm.lat if 'lat' in sliced_gcm else data_var.lat
            target_lev = sliced_gcm.lev if 'lev' in sliced_gcm else data_var.lev
            
            # Interpolate va to match the target latitude grid
            if var_name == 'va' and 'lat' in data_var.coords and 'lat' in target_lat.coords:
                data_var = data_var.interp(lat=target_lat)

            # Assign the adjusted longitude values to ua or va
            data_var = data_var.assign_coords(
                lon = target_lon, lat = target_lat, lev = target_lev
            )
        # Add the processed variable to the dataset
        sliced_gcm[var_name] = data_var
    sliced_gcm_all.append(sliced_gcm)

In [11]:
print(sliced_gcm_all)
print(data_var)

[<xarray.Dataset> Size: 3MB
Dimensions:  (time: 46752, lat: 1, lon: 3)
Coordinates:
  * time     (time) datetime64[ns] 374kB 1958-01-01T06:00:00 ... 1990-01-01
    lev      float64 8B 20.0
  * lat      (lat) float64 8B -10.0
  * lon      (lon) float64 24B 129.4 131.2 133.1
Data variables:
    hus      (time, lat, lon) float32 561kB dask.array<chunksize=(31, 1, 3), meta=np.ndarray>
    ta       (time, lat, lon) float32 561kB dask.array<chunksize=(31, 1, 3), meta=np.ndarray>
    ua       (time, lat, lon) float32 561kB dask.array<chunksize=(31, 1, 3), meta=np.ndarray>
    va       (time, lat, lon) float32 561kB dask.array<chunksize=(31, 1, 3), meta=np.ndarray>]
<xarray.DataArray 'va' (time: 46752, lat: 1, lon: 3)> Size: 561kB
dask.array<transpose, shape=(46752, 1, 3), dtype=float32, chunksize=(31, 1, 3), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 374kB 1958-01-01T06:00:00 ... 1990-01-01
    lev      float64 8B 20.0
  * lon      (lon) float64 24B 129.4 131.2 1

In [ ]:
    if config.save_bc_output:
        # save the bias corrected data # from input gcm or obs to target gcm
        ds_corrected.to_netcdf(
            f"{out_path}/bc_corrected_{level}_{infor}_{gname}_{period}_{cinfor}_{sinfor}_{version}.nc"
        )

    if config.draw_figure:
        ds_corrected_day = assign_w_day(ds_corrected, bc_boundary)

        print("Drawing figures")
        if config.sub_daily_correction:
            print("K-S test has been included")
            statistics = AnalysisBC(
                daily_gcm,
                daily_obs,
                ds_corrected_day,
                variables,
                config.out_figure_path,
                kstest=True,
            )
        if config.sub_daily_correction == False:
            print("K-S test has not been included")
            statistics = AnalysisBC(
                daily_gcm,
                daily_obs,
                ds_corrected_day,
                variables,
                config.out_figure_path,
                kstest=False,
            )
        statistics.figure_atmos()
        print("Finish 3d field")

        if (level == 1) and (no_of_variables >= 4):
            statistics.figure_surface()
            print("Finish 2d field")

if config.reformat_to_original:
    print("Start reformatting")

    reformat_and_save_3d(
        out_path,
        tlevel,
        startyear_h,
        endyear_h,
        variables,
        variables,
        out_path,
        infor,
        gname,
        period,
        cinfor,
        sinfor,
    )
    print("Finish 3D reformatting")

    if (level == 1) and (no_of_variables >= 4):
        reformat_and_save_2d(
            out_path,
            startyear_h,
            endyear_h,
            variables,
            variables,
            out_path,
            infor,
            gname,
            period,
            cinfor,
            sinfor,
            startyear_h,
            endyear_h,
        )
        print("Finish 2D reformatting")